# 03 — Targeted transition dataset v1.1

Questo notebook costruisce il dataset diagnostico mirato per eventi dendritici, branching, recovery e stochasticità sinaptica. Il teacher Hay, i 642 segmenti, i `.mod`, i pesi, il burn-in, CVode, il boundary step da 1 ms e lo stato canonico restano invariati.

La decisione metodologica centrale è causale: `U_realized` viene calcolato prima del macro-step mediante una replay esatta delle equazioni `NET_RECEIVE` e di uno stream Random123 clonato, mentre il teacher continua a usare le sinapsi originali immutate. Al bordo successivo gli stati `NetCon` e la posizione RNG devono coincidere, mentre gli stati continui A/B integrati da CVODE devono preferire il percorso dell'esito reale rispetto al controfattuale ottenuto invertendo successo/fallimento. Questi confronti usano il futuro solo per validazione: `S_(t+1)` non costruisce mai l'input. La procedura evita di assumere un ordine non garantito tra callback Python e `NetCon` allo stesso timestamp. Le classi NMDA sono gerarchiche: ogni plateau è anche un NMDA spike locale sostenuto, mentre `nmda_plateau` richiede almeno 10 ms.

**Protezione degli artefatti:** se una generazione o un replay sono già terminati, non usare `Run All`, non impostare flag di reset e non cancellare `OUTPUT_DIR`. Per la ricertificazione si aggiorna il codice nella sessione viva e si esegue soltanto la sezione 9; una nuova generazione deve usare una nuova `HAYFLOW_OUTPUT_DIR`.

## 1. Checkout riproducibile e teacher separato

In [ ]:
import importlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

ELM_REPOSITORY = "https://github.com/Zagred47/giada.git"
ELM_REF = os.environ.get("HAYFLOW_ELM_REF", "main")
TEACHER_REPOSITORY = "https://github.com/SelfishGene/neuron_as_deep_net.git"
TEACHER_COMMIT = "074c4666300a8ad246601dab179a97a6942f0f29"
NOTEBOOK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd().resolve()
WORKSPACE = NOTEBOOK_ROOT / "hayflow_workspace"
WORKSPACE.mkdir(parents=True, exist_ok=True)

def run(command, cwd=None):
    print("+", " ".join(map(str, command)), flush=True)
    subprocess.run(list(map(str, command)), cwd=cwd, check=True)

elm_override = os.environ.get("HAYFLOW_ELM_REPO")
mounted = [Path(elm_override).expanduser()] if elm_override else []
mounted.extend([Path.cwd(), *Path.cwd().parents])
ELM_REPO = next((p.resolve() for p in mounted if (p / "src" / "hayflow_teacher").is_dir()), None)
if ELM_REPO is None:
    ELM_REPO = WORKSPACE / "elmneuron"
    if not (ELM_REPO / ".git").is_dir():
        run(["git", "clone", ELM_REPOSITORY, ELM_REPO])
    run(["git", "fetch", "origin", ELM_REF], cwd=ELM_REPO)
    run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=ELM_REPO)
TEACHER_REPO = Path(os.environ.get("HAYFLOW_TEACHER_REPO", ELM_REPO.parent / "neuron_as_deep_net")).expanduser().resolve()
if not (TEACHER_REPO / ".git").is_dir():
    run(["git", "clone", TEACHER_REPOSITORY, TEACHER_REPO])
run(["git", "checkout", "--detach", TEACHER_COMMIT], cwd=TEACHER_REPO)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=TEACHER_REPO, text=True).strip() == TEACHER_COMMIT
os.environ["HAYFLOW_ELM_REPO"] = str(ELM_REPO)
os.environ["HAYFLOW_TEACHER_REPO"] = str(TEACHER_REPO)
print("Owned repository:", ELM_REPO)
print("Canonical teacher:", TEACHER_REPO)

## 2. Dipendenze e compilazione dei MOD originali

In [ ]:
run([sys.executable, "-m", "pip", "install", "--quiet", "neuron==8.2.7", "numpy", "pandas", "matplotlib", "h5py", "pyarrow", "pyyaml"])
SIMULATION_DIR = TEACHER_REPO / "L5PC_NEURON_simulation"
compiled = list(SIMULATION_DIR.rglob("libnrnmech.so"))
if not compiled:
    nrnivmodl = shutil.which("nrnivmodl") or str(Path(sys.executable).parent / "nrnivmodl")
    run([nrnivmodl, "mods"], cwd=SIMULATION_DIR)
assert list(SIMULATION_DIR.rglob("libnrnmech.so")), "MOD compilation failed"
sys.path.insert(0, str(ELM_REPO))
for name in tuple(sys.modules):
    if name == "src.hayflow_teacher" or name.startswith("src.hayflow_teacher.") or name == "src.hayflow_data" or name.startswith("src.hayflow_data."):
        sys.modules.pop(name, None)
importlib.invalidate_caches()
print("Teacher mechanisms compiled without source changes.")

## 3. Calibrazione dendritica 01b come provenienza immutabile

In [ ]:
calibration_override = os.environ.get("HAYFLOW_CALIBRATION_SOURCE")
calibration_candidates = [Path(calibration_override).expanduser()] if calibration_override else []
calibration_candidates.extend([
    Path("/kaggle/input/datasets/alessandrobelli/hayflow-dendritic-protocol-calibration/hayflow_dendritic_protocol_calibration"),
    NOTEBOOK_ROOT / "hayflow_dendritic_protocol_calibration",
    NOTEBOOK_ROOT / "hayflow_dendritic_protocol_calibration.zip",
])
if Path("/kaggle/input").is_dir():
    calibration_candidates.extend(Path("/kaggle/input").rglob("hayflow_dendritic_protocol_calibration.zip"))
    calibration_candidates.extend(path.parent for path in Path("/kaggle/input").rglob("selected_dendritic_protocols.json"))
CALIBRATION_SOURCE = next((path.resolve() for path in calibration_candidates if path.exists()), None)
assert CALIBRATION_SOURCE is not None, "Calibrazione 01b non trovata. Imposta HAYFLOW_CALIBRATION_SOURCE."
print("Calibration source:", CALIBRATION_SOURCE)

## 4. Manifest, contratto 1.1 e burn-in misurato

In [ ]:
import yaml
from IPython.display import display
from src.hayflow_data import BurnInCriteria
from src.hayflow_teacher import TargetedDiagnosticDatasetSession, expected_audit_hashes

BASE_CONFIG_PATH = ELM_REPO / "configs" / "hayflow" / "transition_dataset_diagnostic.yml"
V1_CONFIG_PATH = ELM_REPO / "configs" / "hayflow" / "diagnostic_dataset_v1.yml"
TARGET_CONFIG_PATH = ELM_REPO / "configs" / "hayflow" / "targeted_transition_dataset_v1_1.yml"
base_config = yaml.safe_load(BASE_CONFIG_PATH.read_text(encoding="utf-8"))
target_config = yaml.safe_load(TARGET_CONFIG_PATH.read_text(encoding="utf-8"))
OUTPUT_DIR = Path(os.environ.get("HAYFLOW_OUTPUT_DIR", NOTEBOOK_ROOT / "artifacts" / "diagnostic_dataset_v1_1")).expanduser().resolve()
if OUTPUT_DIR.exists():
    raise RuntimeError(
        f"Output già presente e protetto: {OUTPUT_DIR}. Non usare Run All e non cancellarlo per ricertificare un replay completato: aggiorna il codice nella sessione viva e riesegui soltanto la sezione 9. Per una nuova generazione imposta HAYFLOW_OUTPUT_DIR su una cartella nuova."
    )
session = TargetedDiagnosticDatasetSession(
    ELM_REPO, TEACHER_REPO,
    calibration_source=CALIBRATION_SOURCE,
    dataset_config_path=TARGET_CONFIG_PATH,
    output_dir=OUTPUT_DIR,
    seed=base_config["runtime"]["seed"],
    expected_teacher_hashes=expected_audit_hashes(),
    native_snapshot_stride=target_config["storage"]["native_snapshot_stride_ms"],
)
teacher_report = session.prepare_teacher()
contract_report = session.prepare_targeted_contract()
burnin_config = dict(base_config["burnin"])
burnin_config["slow_mechanisms"] = tuple(burnin_config["slow_mechanisms"])
criteria = BurnInCriteria(**burnin_config)
burnin_report = session.run_burn_in(criteria)
current_smoke = session.run_somatic_current_smoke_test(**base_config["somatic_current"]["smoke_test"])
candidate_currents = base_config["somatic_current"]["calibration"]["candidate_amplitudes_na"]
session.calibrate_somatic_spike_current(candidate_amplitudes_na=candidate_currents)
session.calibrate_somatic_single_spike_current(candidate_amplitudes_na=candidate_currents)
display({"teacher": teacher_report, "contract": contract_report, "burnin": burnin_report})
assert teacher_report["segment_count"] == 642
assert contract_report["core_state_width"] == 17220
assert contract_report["privileged_state_width"] == 9182
assert burnin_report["converged"] and current_smoke["valid"]

## 5. Pilot causale del rilascio

Questa è la cella metodologicamente decisiva. Verifica stesso snapshot + stesso Random123 + stesso schedule ⇒ stesso release outcome e stessa transizione. Controlla inoltre che il campione Random123 previsto e il salto diretto degli stati sinaptici concordino.

In [ ]:
release_pilot = session.run_causal_release_pilot(
    transition_count=target_config["release_contract"]["pilot_transition_count"]
)
display(release_pilot)
assert release_pilot["valid"]
assert not release_pilot["future_state_used"]
assert not release_pilot["instrumentation_changes_teacher_dynamics"]

## 6. Pilot biologico adattivo

Lo sweep cerca separatamente spike assonale/somatico, bAP, Ca spike, NMDA spike breve e plateau. Per il bAP seleziona automaticamente l'assistenza dendritica più forte che resta senza eventi e almeno 2 mV sotto soglia in tutti i seed, quindi esegue tre bracci controfattuali: corrente somatica sola, assistenza dendritica sola e combinazione. Un bAP entra nella selezione soltanto se l'assistenza isolata resta subthreshold, lo spike assonale/somatico precede il trunk e nessun evento Ca/NMDA o attraversamento distale precede il trunk. Non viene quindi allargata la finestra di 3 ms per far passare candidati ambigui. Stampa progresso ed ETA. Se una classe non ha positivi robusti e hard-negative su tutti i seed, si ferma qui.

In [ ]:
biological_pilot = session.run_targeted_biological_pilot(target_config["biological_pilot"])
display({
    "valid": biological_pilot["valid"],
    "trial_count": biological_pilot["trial_count"],
    "bap_causal_validation": biological_pilot["bap_causal_validation"],
    "adaptive_blockers": biological_pilot["adaptive_brackets"]["blockers"],
})
assert biological_pilot["valid"]

## 7. Piano bilanciato, budget-aware e split specializzati

Il planner parte dal supporto statistico preferito e cerca deterministicamente il massimo supporto nominale compatibile con 10–30 mila transizioni, senza scendere sotto il minimo diagnostico pre-registrato. Include gli split specializzati nel conteggio. Le quote nominali dimensionano il piano; il minimo diagnostico è il gate scientifico sui label realmente osservati con i nuovi seed Random123. I due contratti vengono persistiti separatamente prima di generare gli snapshot.

In [ ]:
from src.hayflow_data import build_budgeted_episode_plan, summarize_independent_support, write_json
support_config = target_config["support_targets"]
preferred = support_config["preferred"]
minimum = support_config["minimum_diagnostic"]
preferred_positive = {split: int(values["positive_per_class"]) for split, values in preferred.items()}
preferred_negative = {split: int(values["hard_negative_per_class"]) for split, values in preferred.items()}
minimum_positive = {split: int(values["positive_per_class"]) for split, values in minimum.items()}
minimum_negative = {split: int(values["hard_negative_per_class"]) for split, values in minimum.items()}
transition_minimum, transition_maximum = map(int, target_config["storage"]["target_transitions"])
protocols, planned_episodes, budget_report = build_budgeted_episode_plan(
    session.targeted_recipe_catalog,
    preferred_positive_targets=preferred_positive,
    preferred_hard_negative_targets=preferred_negative,
    minimum_positive_targets=minimum_positive,
    minimum_hard_negative_targets=minimum_negative,
    minimum_transition_count=transition_minimum,
    maximum_transition_count=transition_maximum,
    search_steps=int(support_config["budget_search_steps"]),
)
write_json(OUTPUT_DIR / "planning_budget_report.json", budget_report)
planned_transitions = sum(row.duration_ms for row in protocols)
planned_support = summarize_independent_support(planned_episodes)
pre_snapshot_report = session.accept_targeted_protocol_plan(
    protocols,
    pilot_report=biological_pilot,
    positive_support_targets=budget_report["effective_positive_targets"],
    hard_negative_support_targets=budget_report["effective_hard_negative_targets"],
    minimum_positive_support_targets=budget_report["minimum_positive_targets"],
    minimum_hard_negative_support_targets=budget_report["minimum_hard_negative_targets"],
    require_snapshot_bank=False,
)
display({"budget": budget_report, "pre_snapshot_blockers": pre_snapshot_report["blockers"]})
if not pre_snapshot_report["valid"]:
    raise RuntimeError("Piano respinto prima degli snapshot: " + "; ".join(pre_snapshot_report["blockers"]))
snapshot_bank_report = session.prepare_snapshot_bank(protocols, conditioning_ms=4)
plan_report = session.accept_targeted_protocol_plan(
    protocols,
    pilot_report=biological_pilot,
    positive_support_targets=budget_report["effective_positive_targets"],
    hard_negative_support_targets=budget_report["effective_hard_negative_targets"],
    minimum_positive_support_targets=budget_report["minimum_positive_targets"],
    minimum_hard_negative_support_targets=budget_report["minimum_hard_negative_targets"],
    require_snapshot_bank=True,
)
plan_summary = {
    key: plan_report[key]
    for key in (
        "valid", "blockers", "trajectory_count", "transition_count",
        "snapshot_count", "required_splits", "observed_splits",
        "seed_split_leaks", "snapshot_split_leaks",
        "heldout_branches", "planned_support_validation",
    )
}
snapshot_summary = {
    key: snapshot_bank_report[key]
    for key in ("valid", "snapshot_count", "conditioning_ms", "split_specific")
}
display({"plan": plan_summary, "snapshot_bank": snapshot_summary})
if not plan_report["valid"]:
    raise RuntimeError("Protocol plan non valido: " + "; ".join(plan_report["blockers"]))

## 8. Generazione completa

Questa è la cella lunga. Il tracker mostra transizioni completate, traiettoria corrente ed ETA. Al termine viene mostrato immediatamente il supporto osservato contro il minimo diagnostico e contro la quota nominale: se il minimo non è raggiunto, il notebook si ferma qui e non spreca tempo nel replay. Gli artefatti statici vengono indicizzati con SHA-256 prima della validazione. Il dataset rimane diagnostico: 10–30 mila transizioni.

In [ ]:
dataset_manifest = session.generate_dataset(protocols)
dataset_card = json.loads((OUTPUT_DIR / "dataset_card.json").read_text(encoding="utf-8"))
generation_summary = {k: dataset_manifest[k] for k in ("schema_version", "trajectory_count", "transition_count", "event_count", "size_estimate")}
generation_summary["minimum_support_validation"] = dataset_card["minimum_support_validation"]
generation_summary["planned_target_attainment"] = dataset_card["planned_target_attainment"]
display(generation_summary)
if not dataset_card["minimum_support_validation"]["valid"]:
    raise RuntimeError("Supporto osservato sotto il minimo diagnostico; il replay non viene avviato: " + str(dataset_card["minimum_support_validation"]["failures"]))

## 9. Validazione a gate e report finale

Prima vengono eseguiti i controlli economici su supporto, split, label, release, provenienza e hash delle tabelle. Solo se sono verdi parte il replay esaustivo. Un replay già completato può essere riutilizzato esclusivamente dopo verifica SHA-256 dello stesso HDF5; un replay nuovo viene salvato atomicamente appena termina, prima delle elaborazioni finali, così un riavvio non lo ripete. La quota nominale non raggiunta resta un warning, mentre soltanto il minimo diagnostico pre-registrato è bloccante.

In [ ]:
validation_report = session.validate_dataset_v1_1(raise_on_failure=False)
dataset_card = json.loads((OUTPUT_DIR / "dataset_card.json").read_text(encoding="utf-8"))
display({"validation": validation_report, "dataset_card": dataset_card})
assert validation_report["valid"], validation_report["blockers"]

## 10. Download ZIP con Base64 + Blob

Questa è la procedura di download stabile usata nel progetto: `make_archive`, trasferimento Base64 al browser e vero link temporaneo `<a download>`. Per artefatti molto grandi la codifica richiede memoria aggiuntiva nel kernel e nel browser.

In [ ]:
from base64 import b64encode
from shutil import make_archive
from IPython.display import Javascript, display

artifact_dir = OUTPUT_DIR.resolve()
zip_base = NOTEBOOK_ROOT / "hayflow_targeted_transition_dataset_v1_1"
zip_path = Path(make_archive(str(zip_base), "zip", root_dir=artifact_dir.parent, base_dir=artifact_dir.name))
payload = b64encode(zip_path.read_bytes()).decode("ascii")
filename = zip_path.name
print(f"ZIP pronto: {zip_path} ({zip_path.stat().st_size / 1024**3:.2f} GiB)")
display(Javascript(f'''
(() => {{
  const binary = atob("{payload}");
  const bytes = new Uint8Array(binary.length);
  for (let i = 0; i < binary.length; i++) bytes[i] = binary.charCodeAt(i);
  const blob = new Blob([bytes], {{type: "application/zip"}});
  const url = URL.createObjectURL(blob);
  const link = document.createElement("a");
  link.href = url;
  link.download = "{filename}";
  document.body.appendChild(link);
  link.click();
  link.remove();
  setTimeout(() => URL.revokeObjectURL(url), 60000);
}})();
'''))